# 0. Đánh giá Tuần 4 - phần tích hợp của Nhân

Notebook này tích hợp các đầu ra đánh giá đã có của Lợi và Danh với phần benchmark hệ thống, ma trận đánh đổi và khuyến nghị mô hình của Nhân. Notebook **không huấn luyện lại mô hình**, không sửa pipeline và không ghi đè ba artifact Tuần 3.

Quy ước dữ liệu: các mô hình đã được huấn luyện trên 32 đặc trưng sau tiền xử lý, vì vậy notebook dùng trực tiếp `data/processed/X_test.parquet`. Không gọi lại `preprocessing_pipeline.pkl`, tránh biến đổi hai lần.

## 1. Môi trường và khả năng tái lập

In [1]:
from pathlib import Path
import hashlib
import json
import platform
import sys

import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Không xác định được thư mục gốc của dự án.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation.benchmark import (
    build_tradeoff_matrix,
    load_preprocessed_test_features,
    load_saved_models,
    recommend_provisional_champion,
    sha256_file,
    verify_model_contract,
)

environment = {
    "Python": platform.python_version(),
    "scikit-learn": sklearn.__version__,
    "pandas": pd.__version__,
    "NumPy": np.__version__,
    "Hệ điều hành": platform.platform(),
}
pd.Series(environment, name="Giá trị").to_frame()

,Giá trị
Python,3.13.0
scikit-learn,1.9.1
pandas,2.2.3
NumPy,2.1.2
Hệ điều hành,Windows-11-10.0.26200-SP0


## 2. Dữ liệu test và nguyên tắc không huấn luyện

Tập test chỉ được dùng để đánh giá cuối cùng, đo tốc độ dự đoán và kiểm tra artifact. Không có lệnh `fit`, tuning hoặc chọn siêu tham số trong notebook. Nhãn test không tham gia benchmark tốc độ.

In [2]:
X_test = load_preprocessed_test_features(
    PROJECT_ROOT / "data" / "processed" / "X_test.parquet"
)
y_test = pd.read_parquet(
    PROJECT_ROOT / "data" / "processed" / "y_test.parquet"
).iloc[:, 0]

assert len(X_test) == len(y_test)
assert X_test.shape == (2233, 32)
assert y_test.name == "deposit"
assert not X_test.isna().any().any()
assert np.isfinite(X_test.to_numpy(dtype=float)).all()

pd.DataFrame({
    "Đối tượng": ["X_test", "y_test"],
    "Số dòng": [len(X_test), len(y_test)],
    "Số cột": [X_test.shape[1], 1],
    "Vai trò": ["Đặc trưng đã tiền xử lý", "Nhãn đánh giá held-out"],
})

,Đối tượng,Số dòng,Số cột,Vai trò
0,X_test,2233,32,Đặc trưng đã tiền xử lý
1,y_test,2233,1,Nhãn đánh giá held-out


## 3. Kiểm tra hợp đồng đầu vào của các artifact Tuần 3

In [3]:
models = load_saved_models()
contract_rows = []
for family, model in models.items():
    contract = verify_model_contract(model, X_test)
    contract_rows.append({
        "Họ mô hình": family,
        "Estimator": type(model).__name__,
        "Số đặc trưng": contract["n_features_expected"],
        "Tên/thứ tự cột khớp": contract["feature_names_match"],
        "Là pipeline": contract["estimator_is_pipeline"],
        "Benchmark gọi preprocessing": contract["preprocessing_applied_by_benchmark"],
    })
contract_df = pd.DataFrame(contract_rows)
display(contract_df)
assert contract_df["Tên/thứ tự cột khớp"].all()
assert not contract_df["Là pipeline"].any()
assert not contract_df["Benchmark gọi preprocessing"].any()

,Họ mô hình,Estimator,Số đặc trưng,Tên/thứ tự cột khớp,Là pipeline,Benchmark gọi preprocessing
0,linear,LogisticRegression,32,True,False,False
1,tree,RandomForestClassifier,32,True,False,False
2,nonlinear,SVC,32,True,False,False


## 4. Đầu vào Week 4 kế thừa từ đồng đội

In [4]:
component_status = pd.DataFrame([
    {
        "Phụ trách": "Lợi",
        "Thành phần": "src/evaluation/metrics.py",
        "Có trên main": (PROJECT_ROOT / "src/evaluation/metrics.py").exists(),
        "Cách tích hợp": "Đọc evaluation_benchmark.csv và test_predictions.parquet",
    },
    {
        "Phụ trách": "Trí",
        "Thành phần": "src/evaluation/plots.py",
        "Có trên main": (PROJECT_ROOT / "src/evaluation/plots.py").exists(),
        "Cách tích hợp": "Không có đầu vào; không tạo placeholder thay thế",
    },
    {
        "Phụ trách": "Danh",
        "Thành phần": "src/evaluation/error_analysis.py",
        "Có trên main": (PROJECT_ROOT / "src/evaluation/error_analysis.py").exists(),
        "Cách tích hợp": "Đọc outcome_summary.csv và head_to_head.csv",
    },
])
component_status

,Phụ trách,Thành phần,Có trên main,Cách tích hợp
0,Lợi,src/evaluation/metrics.py,True,Đọc evaluation_benchmark.csv và test_predictio...
1,Trí,src/evaluation/plots.py,False,Không có đầu vào; không tạo placeholder thay thế
2,Danh,src/evaluation/error_analysis.py,True,Đọc outcome_summary.csv và head_to_head.csv


## 5. Kết quả định lượng của Lợi

Các chỉ số dưới đây được đọc từ artifact Week 4 của Lợi. Notebook không triển khai lại `metrics.py` và không ghi đè bảng dự đoán của đồng đội.

In [5]:
evaluation = pd.read_csv(PROJECT_ROOT / "reports/evaluation_benchmark.csv")
expected_families = {"dummy", "linear", "tree", "nonlinear"}
assert set(evaluation["Model_Key"]) == expected_families

evaluation_view = evaluation[[
    "Model_Name",
    "CV_Macro_F1 (W3)",
    "Test_Macro_F1 (W4)",
    "Generalization_Gap",
    "Test_Accuracy",
    "Test_Precision",
    "Test_Recall",
    "Test_ROC_AUC",
    "Test_PR_AUC",
]].copy()
evaluation_view.round(4)

,Model_Name,CV_Macro_F1 (W3),Test_Macro_F1 (W4),Generalization_Gap,Test_Accuracy,Test_Precision,Test_Recall,Test_ROC_AUC,Test_PR_AUC
0,Dummy Baseline (Most Frequent),0.3448,0.3448,0.0000,0.5262,0.0000,0.0000,0.5000,0.4738
1,Logistic Regression (Tuned),0.6960,0.7098,-0.0137,0.7147,0.7384,0.6163,0.7605,0.7654
2,Random Forest (Tuned),0.7233,0.7263,-0.0030,0.7313,0.7626,0.6285,0.7747,0.7774
3,SVC RBF (Tuned),0.7178,0.7185,-0.0007,0.7268,0.7835,0.5851,0.7729,0.7727


## 6. Benchmark hệ thống của Nhân

Kết quả được tạo bởi `src/evaluation/benchmark.py` với warm-up và nhiều lần đo bằng `perf_counter_ns`. Artifact lưu cả trung vị, p95, độ lệch chuẩn, môi trường phần cứng/phần mềm và SHA-256. Có thể tái chạy từ thư mục gốc bằng `py -3.13 -m src.evaluation.benchmark`; notebook đọc snapshot đã xác nhận để tránh thay đổi số đo mỗi lần mở.

In [6]:
benchmark_path = PROJECT_ROOT / "reports/week4_system_benchmark.json"
benchmark_payload = json.loads(benchmark_path.read_text(encoding="utf-8"))

benchmark_rows = []
for row in benchmark_payload["models"]:
    benchmark_rows.append({
        "Họ mô hình": row["family"],
        "Latency 1 mẫu median (ms)": row["single_sample_latency_ms"]["median"],
        "Latency 1 mẫu p95 (ms)": row["single_sample_latency_ms"]["p95"],
        "Batch 2.233 mẫu median (ms)": row["batch_latency_ms"]["median"],
        "Throughput median (mẫu/s)": row["batch_throughput_samples_per_second"]["median"],
        "Kích thước (MB)": row["model_size_mb"],
        "Train time ghi nhận (s)": row["recorded_training_seconds"],
        "Tuning time ghi nhận (s)": row["recorded_tuning_seconds"],
    })
benchmark_df = pd.DataFrame(benchmark_rows)
display(benchmark_df.round(4))

pd.Series(benchmark_payload["environment"], name="Giá trị").to_frame()

,Họ mô hình,Latency 1 mẫu median (ms),Latency 1 mẫu p95 (ms),Batch 2.233 mẫu median (ms),Throughput median (mẫu/s),Kích thước (MB),Train time ghi nhận (s),Tuning time ghi nhận (s)
0,linear,0.3490,0.3818,0.4244,5.261546e+06,0.0026,NaN,NaN
1,tree,41.1086,43.6485,61.5356,3.628794e+04,41.5018,NaN,NaN
2,nonlinear,0.7476,1.0237,631.8272,3.534194e+03,1.5228,5.684,320.0


,Giá trị
timestamp_utc,2026-10-20T06:12:21.240116+00:00
operating_system,Windows-11-10.0.26200-SP0
machine,AMD64
processor,"AMD64 Family 25 Model 116 Stepping 1, Authenti..."
logical_cpu_count,16
python_version,3.13.0
python_executable,C:\Users\Admin\AppData\Local\Programs\Python\P...
numpy_version,2.1.2
pandas_version,2.2.3
scikit_learn_version,1.9.1


## 7. Ma trận đánh đổi

In [7]:
tradeoff = build_tradeoff_matrix(
    benchmark_payload,
    evaluation_csv=PROJECT_ROOT / "reports/evaluation_benchmark.csv",
    output_json=None,
)
stored_tradeoff = json.loads(
    (PROJECT_ROOT / "reports/week4_tradeoff_matrix.json").read_text(encoding="utf-8")
)
pd.testing.assert_frame_equal(
    pd.DataFrame(tradeoff),
    pd.DataFrame(stored_tradeoff),
    check_exact=False,
    rtol=1e-12,
    atol=1e-12,
)

tradeoff_view = pd.DataFrame(tradeoff)[[
    "model_name",
    "test_macro_f1",
    "generalization_gap_cv_minus_test",
    "test_precision_yes",
    "test_recall_yes",
    "single_sample_latency_ms_median",
    "batch_throughput_samples_per_second_median",
    "model_size_mb",
    "interpretability",
]].copy()
tradeoff_view.columns = [
    "Mô hình", "Test Macro-F1", "Gap CV-Test", "Precision yes", "Recall yes",
    "Latency median (ms)", "Throughput (mẫu/s)", "Size (MB)", "Khả năng giải thích"
]
tradeoff_view.round(4)

,Mô hình,Test Macro-F1,Gap CV-Test,Precision yes,Recall yes,Latency median (ms),Throughput (mẫu/s),Size (MB),Khả năng giải thích
0,Logistic Regression (Tuned),0.7098,-0.0137,0.7384,0.6163,0.3490,5.261546e+06,0.0026,High
1,Random Forest (Tuned),0.7263,-0.0030,0.7626,0.6285,41.1086,3.628794e+04,41.5018,Medium
2,SVC RBF (Tuned),0.7185,-0.0007,0.7835,0.5851,0.7476,3.534194e+03,1.5228,Low


**Diễn giải đánh đổi.** Logistic Regression có artifact nhỏ nhất, độ trễ thấp nhất và khả năng giải thích trực tiếp. Random Forest đạt Macro-F1 và Recall lớp `yes` cao nhất; throughput batch vẫn cao nhưng artifact lớn và độ trễ đơn mẫu cao. SVC có Precision lớp `yes` cao nhất và artifact nhỏ hơn Random Forest, nhưng throughput batch thấp nhất và khó giải thích.

## 8. Kết quả phân tích lỗi của Danh

In [8]:
outcome_summary = pd.read_csv(
    PROJECT_ROOT / "reports/error_analysis/outcome_summary.csv"
)
head_to_head = pd.read_csv(
    PROJECT_ROOT / "reports/error_analysis/head_to_head.csv"
)
display(outcome_summary.round(4))
display(head_to_head.round(4))

,model,TP,TN,FP,FN,n_errors,error_rate,FN_share_of_errors,errors_near_boundary_share,errors_confident_share,median_error_margin
0,Logistic Regression,652,944,231,406,637,0.2853,0.6374,0.4380,0.1099,0.1192
1,Random Forest,665,968,207,393,600,0.2687,0.6550,0.3000,0.1683,0.1660
2,SVC RBF,619,1004,171,439,610,0.2732,0.7197,0.1869,0.0475,0.2039


,model_a,model_b,both_correct,only_a_correct,only_b_correct,both_wrong,mcnemar_p_value
0,Random Forest,Logistic Regression,1459,174,137,463,0.0410
1,SVC RBF,Logistic Regression,1484,139,112,498,0.1006
2,Random Forest,SVC RBF,1520,113,103,497,0.5404


Danh ghi nhận Random Forest có 600 lỗi trên 2.233 mẫu, thấp hơn Logistic Regression (637) và SVC RBF (610). So sánh McNemar cho thấy Random Forest tốt hơn Logistic Regression với `p = 0,041`, nhưng khác biệt Random Forest - SVC không có ý nghĩa thống kê (`p = 0,540`). Phần lớn lỗi vẫn là false negative; các điểm mù đáng chú ý gồm khách hàng chưa từng được liên hệ và `contact=unknown`.

## 9. Khuyến nghị champion tạm thời

In [9]:
recommendation = json.loads(
    (PROJECT_ROOT / "reports/week4_champion_recommendation.json").read_text(
        encoding="utf-8"
    )
)
recomputed_recommendation = recommend_provisional_champion(tradeoff)
assert recommendation["selected_family"] == recomputed_recommendation["selected_family"]
assert recommendation["status"] == "provisional_recommendation_pending_team_confirmation"

champion_path = PROJECT_ROOT / "models/champion_model.pkl"
selected_source = PROJECT_ROOT / recommendation["artifact_copy"]["source_artifact"]
assert sha256_file(champion_path) == sha256_file(selected_source)

champion_model = joblib.load(champion_path)
sample_prediction = champion_model.predict(X_test.iloc[:10])
assert len(sample_prediction) == 10

pd.Series({
    "Trạng thái": recommendation["status"],
    "Mô hình đề xuất": recommendation["selected_model"],
    "Bối cảnh": recommendation["selection_context"],
    "Macro-F1 test": recommendation["evidence"]["test_macro_f1"],
    "Artifact giống nguồn theo SHA-256": True,
    "Dự đoán mẫu thành công": len(sample_prediction),
}, name="Giá trị").to_frame()

,Giá trị
Trạng thái,provisional_recommendation_pending_team_confir...
Mô hình đề xuất,Random Forest (Tuned)
Bối cảnh,offline_or_batch_bank_marketing_scoring
Macro-F1 test,0.726263
Artifact giống nguồn theo SHA-256,True
Dự đoán mẫu thành công,10


Random Forest là **khuyến nghị tạm thời**, chưa phải quyết định đã được cả nhóm xác nhận. Lựa chọn phù hợp với bối cảnh chấm điểm chiến dịch theo batch: mô hình đứng đầu Macro-F1 và Recall lớp `yes`, gap CV-test nhỏ, đồng thời throughput đo được đủ cao trên máy thử nghiệm. Đổi lại, mô hình lớn 41,50 MB, độ trễ đơn mẫu khoảng 41 ms và chỉ có mức giải thích trung bình. Nếu hệ thống tương lai ưu tiên footprint, latency hoặc giải thích, Logistic Regression là phương án thay thế rõ ràng; nếu ưu tiên Precision lớp `yes`, SVC cần được cân nhắc.

## 10. Kiểm tra tính toàn vẹn và giới hạn tích hợp

In [10]:
predictions = pd.read_parquet(PROJECT_ROOT / "reports/test_predictions.parquet")
assert len(predictions) == len(X_test)

for row in benchmark_payload["models"]:
    artifact = PROJECT_ROOT / row["artifact_path"]
    assert sha256_file(artifact) == row["artifact_sha256"]

benchmark_source = (PROJECT_ROOT / "src/evaluation/benchmark.py").read_text(
    encoding="utf-8"
)
assert ".fit(" not in benchmark_source

validation = pd.DataFrame([
    {"Kiểm tra": "Test rows khớp prediction rows", "Kết quả": "PASS"},
    {"Kiểm tra": "Ba artifact Tuần 3 khớp SHA-256", "Kết quả": "PASS"},
    {"Kiểm tra": "Champion là bản sao byte-for-byte", "Kết quả": "PASS"},
    {"Kiểm tra": "benchmark.py không gọi .fit", "Kết quả": "PASS"},
    {"Kiểm tra": "plots.py của Trí có trên main", "Kết quả": "MISSING"},
])
validation

,Kiểm tra,Kết quả
0,Test rows khớp prediction rows,PASS
1,Ba artifact Tuần 3 khớp SHA-256,PASS
2,Champion là bản sao byte-for-byte,PASS
3,benchmark.py không gọi .fit,PASS
4,plots.py của Trí có trên main,MISSING


## 11. Kết luận Tuần 4

- Kết quả hiệu năng định lượng của Lợi và phân tích lỗi của Danh đã được tích hợp từ artifact hiện có, không viết lại module của đồng đội.
- Đầu ra biểu đồ của Trí chưa có trên `main`, nên notebook không tạo biểu đồ thay thế và ghi nhận đây là điểm tích hợp còn thiếu.
- Benchmark đo prediction trên đúng dữ liệu đã tiền xử lý, với warm-up và repeated measurements; không huấn luyện hoặc biến đổi lại dữ liệu.
- Random Forest được đề xuất tạm thời cho bài toán chấm điểm theo batch. Xác nhận champion cuối cùng vẫn cần quyết định của nhóm và yêu cầu vận hành cụ thể.